# Classification Using Neural Networks

Changing the cost function from the regression notebook allows us to solve classification problems. The classic example to this is classifying handwritten digits from the MNIST dataset, which is what we will do here. 

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401
from fys_stk4155_p1.optimization.optimizers import Adam
from matplotlib.colors import Normalize, TwoSlopeNorm
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
from sklearn.model_selection import train_test_split
from tensorflow.keras.datasets import mnist

from fys_stk4155_p2.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from fys_stk4155_p2.models.logistic import LogisticRegression
from fys_stk4155_p2.nn.activations import LeakyReLU, ReLU, Sigmoid

plt.style.use(["science", "ieee"])

Config:

In [ ]:
FIGURE_DIR = Path("../docs/figures/classification")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_CLASSES = 10
VALIDATION_SIZE = 10_000

OPTIMIZER = Adam
EPOCHS = 50
BATCH_SIZE = 128

# Logistic regression baseline
LOGREG_LEARNING_RATE = [1e-4, 1e-3, 1e-2, 1e-1]
LOGREG_REGULARIZATION_STRENGTH = [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]

# Neural networks
REGULARIZATION_STRENGTH = [0.1, 0.01, 0.001, 0.0001, 0.0]
ACTIVATION = [Sigmoid, ReLU, LeakyReLU]

N_HIDDEN_LAYERS = [4, 8]
N_NODES_PER_LAYER = [40, 80]

Loading and preprocess the dataset. All tuning is done on a validation set split off
the training images; the test set is left untouched until the final evaluation.

In [ ]:
# Load the train and test data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.reshape(len(x_train), -1) / 255.0
x_test = x_test.reshape(len(x_test), -1) / 255.0

x_fit, x_val, y_fit, y_val = train_test_split(
    x_train, y_train, test_size=VALIDATION_SIZE, stratify=y_train, random_state=SEED
)

print(f"Training data shape: {x_fit.shape}, validation data shape: {x_val.shape}")

## Logistic Regression as a Baseline

Logistic regression is the network with no hidden layers: one Dense layer followed by the
softmax and the cross entropy. A network with hidden layers is only worth its cost if it
beats this. As an even simpler reference, we first check the accuracy of always guessing
the most common digit.

In [ ]:
majority_class = np.bincount(y_fit).argmax()
majority_accuracy = accuracy_score(y_val, np.full_like(y_val, majority_class))
print(f"Majority-class ({majority_class}) validation accuracy: {majority_accuracy:.4f}")

### Tuning the learning rate and the penalty

We search a grid of learning rates $\gamma$ and $L_2$ penalties $\lambda$, scored by the
validation accuracy. The penalty multiplies the sum over all $784 \times 10$ squared weights,
while the cross entropy is a mean over the samples, so the useful values of $\lambda$ are
small.

In [ ]:
def fit_logistic_regression(
    learning_rate: float,
    lambda_: float,
    validation_data: tuple[np.ndarray, np.ndarray] | None = None,
) -> tuple[LogisticRegression, list[float]]:
    model = LogisticRegression(
        optimizer=OPTIMIZER(learning_rate=learning_rate),
        regularization="l2",
        lambda_=lambda_,
        seed=SEED,
    )
    history = model.fit(
        X=x_fit,
        y=y_fit,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=validation_data,
    )
    return model, history


rows = []
for learning_rate in LOGREG_LEARNING_RATE:
    for lambda_ in LOGREG_REGULARIZATION_STRENGTH:
        model, _ = fit_logistic_regression(learning_rate, lambda_)
        rows.append(
            {
                "learning_rate": learning_rate,
                "lambda": lambda_,
                "val_accuracy": accuracy_score(y_val, model.predict(x_val)),
            }
        )
logreg_grid = pd.DataFrame(rows)

best = logreg_grid.loc[logreg_grid["val_accuracy"].idxmax()]
BEST_LEARNING_RATE, BEST_LAMBDA = best["learning_rate"], best["lambda"]
print(
    f"Best: gamma = {BEST_LEARNING_RATE:g}, lambda = {BEST_LAMBDA:g}, "
    f"validation accuracy = {best['val_accuracy']:.4f}"
)

In [ ]:
def power_label(value: float) -> str:
    return "0" if value == 0 else f"$10^{{{round(np.log10(value))}}}$"


accuracy_table = logreg_grid.pivot_table(
    index="lambda", columns="learning_rate", values="val_accuracy"
).loc[LOGREG_REGULARIZATION_STRENGTH, LOGREG_LEARNING_RATE]
table = 100 * accuracy_table.to_numpy()
norm = Normalize(vmin=table.min(), vmax=table.max())

fig, ax = plt.subplots(figsize=(3.3, 2.6), constrained_layout=True)
image = ax.imshow(table, cmap="Blues", norm=norm, aspect="auto")
for (i, j), value in np.ndenumerate(table):
    color = "white" if norm(value) > 0.6 else "#1f1e1d"
    ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=7, color=color)
ax.set_xticks(range(len(LOGREG_LEARNING_RATE)), [power_label(v) for v in LOGREG_LEARNING_RATE])
ax.set_yticks(
    range(len(LOGREG_REGULARIZATION_STRENGTH)),
    [power_label(v) for v in LOGREG_REGULARIZATION_STRENGTH],
)
ax.set_xlabel(r"$\gamma$")
ax.set_ylabel(r"$\lambda$")
ax.minorticks_off()
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.colorbar(image, ax=ax, label="Validation accuracy (percent)", shrink=0.9)
fig.savefig(FIGURE_DIR / "logreg_hyperparameters.pdf")
plt.show()

### The selected model

We retrain the best configuration, now tracking the validation cost after every epoch.

In [ ]:
logistic_regression, history = fit_logistic_regression(
    BEST_LEARNING_RATE, BEST_LAMBDA, validation_data=(x_val, y_val)
)

In [ ]:
epochs = np.arange(1, len(history) + 1)

fig, ax = plt.subplots(figsize=(3.3, 2.2))
ax.plot(epochs, history, label="Training")
if logistic_regression.validation_history:
    ax.plot(epochs, logistic_regression.validation_history, label="Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross entropy + L2 penalty")
ax.legend()
fig.savefig(FIGURE_DIR / "logreg_learning_curve.pdf")
plt.show()

In [ ]:
y_pred = logistic_regression.predict(x_val)

logreg_metrics = pd.DataFrame(
    {
        "precision": precision_score(y_val, y_pred, n_classes=N_CLASSES),
        "recall": recall_score(y_val, y_pred, n_classes=N_CLASSES),
        "f1": f1_score(y_val, y_pred, n_classes=N_CLASSES),
    }
).rename_axis("digit")

print(f"Validation accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(f"Macro-averaged F1 score: {logreg_metrics['f1'].mean():.4f}")
logreg_metrics.round(3)

In [ ]:
cm = confusion_matrix(y_val, y_pred, n_classes=N_CLASSES)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(3.3, 3.0), constrained_layout=True)
image = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
for (i, j), value in np.ndenumerate(cm_norm):
    if value >= 0.005:
        color = "white" if value > 0.6 else "#1f1e1d"
        ax.text(j, i, f"{100 * value:.0f}", ha="center", va="center", fontsize=5, color=color)
ax.set_xticks(range(N_CLASSES))
ax.set_yticks(range(N_CLASSES))
ax.set_xlabel("Predicted digit")
ax.set_ylabel("True digit")
ax.minorticks_off()
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.colorbar(image, ax=ax, label="Share of true digit", shrink=0.8)
fig.savefig(FIGURE_DIR / "logreg_confusion_matrix.pdf")
plt.show()

Each column of the weight matrix is a $28 \times 28$ template that the model matches
against the image; red pixels push towards the digit and blue pixels away from it.

In [ ]:
weights = logistic_regression.network.layers[0].weights  # shape (784, 10)
limit = np.abs(weights).max()

fig, axes = plt.subplots(1, 10, figsize=(7, 1.0), constrained_layout=True)
for digit, ax in enumerate(axes):
    image = ax.imshow(
        weights[:, digit].reshape(28, 28),
        cmap="RdBu_r",
        norm=TwoSlopeNorm(vcenter=0, vmin=-limit, vmax=limit),
    )
    ax.set_title(str(digit), fontsize=8)
    ax.axis("off")
fig.colorbar(image, ax=axes, label="Weight", shrink=0.9)
fig.savefig(FIGURE_DIR / "logreg_weights.pdf")
plt.show()

### Comparison with Scikit-Learn

Scikit-Learn minimizes $\frac{1}{2}\lVert W \rVert^2 + C \sum_i L_i$, a sum over the samples
rather than a mean. Dividing by $Cn$ shows that our $\lambda$ corresponds to
$C = 1/(2\lambda n)$, and $\lambda = 0$ to $C = \infty$.

In [ ]:
sklearn_logreg = SklearnLogisticRegression(
    C=1 / (2 * BEST_LAMBDA * len(x_fit)) if BEST_LAMBDA > 0 else np.inf,
    max_iter=1000,
    random_state=SEED,
)
sklearn_logreg.fit(x_fit, y_fit)

print(f"Ours:         {accuracy_score(y_val, y_pred):.4f}")
print(f"Scikit-Learn: {accuracy_score(y_val, sklearn_logreg.predict(x_val)):.4f}")

## Final Evaluation on the Test Set

The test set is used once, here, for the models selected on the validation data.

In [ ]:
test_results = pd.DataFrame(
    {
        "test_accuracy": {
            "Majority class": accuracy_score(y_test, np.full_like(y_test, majority_class)),
            "Logistic regression (ours)": accuracy_score(
                y_test, logistic_regression.predict(x_test)
            ),
            "Logistic regression (Scikit-Learn)": accuracy_score(
                y_test, sklearn_logreg.predict(x_test)
            ),
        }
    }
)
test_results.round(4)